In [91]:
import pandas as pd
import requests
import json
import time
from pathlib import Path

import torch

In [92]:
with open("../data/bhagavad_gita_sft_no_sanskrit.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} conversations")

Loaded 620 conversations


In [93]:
def get_stats(ids):
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts


def merge(ids, pair, new_id):
    new_ids = []
    i = 0

    while i < len(ids):
        if (
            i + 1 < len(ids)
            and ids[i] == pair[0]
            and ids[i + 1] == pair[1]
        ):
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids


# Build token ID -> byte sequence mapping.
def build_vocab(merges):
    vocab = {
        token_id: bytes([token_id])
        for token_id in range(256)
    }

    for pair, new_id in merges.items():
        vocab[new_id] = (
            vocab[pair[0]] + vocab[pair[1]]
        )

    return vocab


def encode(text, merges):
    ids = list(str(text).encode("utf-8"))

    # Earlier learned merges have higher priority.
    merge_ranks = {
        pair: rank
        for rank, pair in enumerate(merges.keys())
    }

    while len(ids) >= 2:
        stats = get_stats(ids)

        # Find pairs that exist in the learned merges.
        valid_pairs = [
            pair
            for pair in stats
            if pair in merges
        ]

        if not valid_pairs:
            break

        # Select the earliest learned pair.
        best_pair = min(
            valid_pairs,
            key=lambda pair: merge_ranks[pair],
        )

        ids = merge(
            ids,
            best_pair,
            merges[best_pair],
        )

    return ids


def decode(ids, vocab):
    byte_sequence = b"".join(
        vocab[token_id]
        for token_id in ids
    )

    return byte_sequence.decode("utf-8")

In [94]:
def load_tokenizer(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Tokenizer file not found: {path.resolve()}"
        )

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    # Restore ordered BPE merge rules.
    merges = {
        (first_id, second_id): new_id
        for first_id, second_id, new_id in data["merges"]
    }

    vocab = build_vocab(merges)

    return merges, vocab, data["vocab_size"]

In [95]:
tokenizer_path = Path("tokenizer/tokenizer/tokenizer.json")
if not tokenizer_path.exists():
    tokenizer_path = Path("..") / tokenizer_path

merges, vocab, vocab_size = load_tokenizer(tokenizer_path)

print("Tokenizer loaded")
print("Vocabulary size:", vocab_size)
print("Number of merges:", len(merges))

Tokenizer loaded
Vocabulary size: 1000
Number of merges: 744


In [96]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder


def build_fast_tokenizer(merges, vocab):
    byte_values = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )

    unicode_values = byte_values.copy()
    extra_index = 0

    for byte_value in range(256):
        if byte_value not in byte_values:
            byte_values.append(byte_value)
            unicode_values.append(256 + extra_index)
            extra_index += 1

    byte_encoder = {
        byte_value: chr(unicode_value)
        for byte_value, unicode_value
        in zip(byte_values, unicode_values)
    }

    def bytes_to_token_string(byte_sequence):
        return "".join(
            byte_encoder[byte_value]
            for byte_value in byte_sequence
        )

    fast_vocab = {
        bytes_to_token_string(byte_sequence): token_id
        for token_id, byte_sequence in vocab.items()
    }

    fast_merges = [
        (
            bytes_to_token_string(vocab[first_id]),
            bytes_to_token_string(vocab[second_id]),
        )
        for first_id, second_id in merges
    ]

    tokenizer = Tokenizer(
        BPE(
            vocab=fast_vocab,
            merges=fast_merges,
        )
    )

    tokenizer.pre_tokenizer = ByteLevel(
        add_prefix_space=False,
        use_regex=False,
    )

    tokenizer.decoder = ByteLevelDecoder()

    return tokenizer


fast_tokenizer = build_fast_tokenizer(
    merges=merges,
    vocab=vocab,
)

print("Fast tokenizer created")
print("Vocabulary size:", fast_tokenizer.get_vocab_size())

Fast tokenizer created
Vocabulary size: 1000


In [97]:
def format_conversation(conversation):
    text = ""

    for message in conversation["messages"]:
        role = message["role"]
        content = message["content"]

        text += f"{role}: {content}\n"

    return text

In [98]:
all_input_ids = []
all_text = ""
for conversation in data:
    text = format_conversation(conversation)
    all_text += text + "<newLINE>"+  "\n" 

    encoding = fast_tokenizer.encode(text)

    all_input_ids.append(encoding.ids)

In [99]:
print("Conversations:", len(data))
print("Encoded conversations:", len(all_input_ids))

Conversations: 620
Encoded conversations: 620


In [100]:
lengths = [len(ids) for ids in all_input_ids]

print("Total conversations:", len(all_input_ids))
print("Minimum tokens:", min(lengths))
print("Maximum tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))

Total conversations: 620
Minimum tokens: 164
Maximum tokens: 3208
Average tokens: 1285.116129032258


In [101]:
import numpy as np

print("P50:", np.percentile(lengths, 50))
print("P75:", np.percentile(lengths, 75))
print("P90:", np.percentile(lengths, 90))
print("P95:", np.percentile(lengths, 95))
print("P99:", np.percentile(lengths, 99))

P50: 1247.0
P75: 1526.25
P90: 1823.3000000000002
P95: 2068.1
P99: 2479.43


In [102]:
import re

CONTEXT_LENGTH = 512
MAX_USER_TOKENS = 350


def token_length(text):
    return len(fast_tokenizer.encode(text).ids)


def format_pair(user_text, assistant_text):
    return (
        f"user: {user_text}\n"
        f"assistant: {assistant_text}"
    )


def shorten_user_prompt(user_text, max_tokens=MAX_USER_TOKENS):
    user_ids = fast_tokenizer.encode(user_text).ids

    if len(user_ids) <= max_tokens:
        return user_text, False

    user_ids = user_ids[-max_tokens:]

    return fast_tokenizer.decode(user_ids).strip(), True


def split_into_sentences(text):
    text = text.strip()

    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    parts = re.split(
        r'\n\s*\n|(?<=[.!?])\s+',
        text
    )

    return [
        part.strip()
        for part in parts
        if part.strip()
    ]


def hard_split_text(text, prefix, context_length=512):
    words = text.split()

    chunks = []
    current_words = []

    for word in words:
        candidate_words = current_words + [word]
        candidate = " ".join(candidate_words)

        if token_length(prefix + candidate) <= context_length:
            current_words.append(word)

        else:
            if current_words:
                chunks.append(" ".join(current_words))

            current_words = [word]

            if token_length(prefix + word) > context_length:
                current_words = []

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def split_assistant_response(
    user_text,
    assistant_text,
    context_length=512
):
    prefix = (
        f"user: {user_text}\n"
        f"assistant: "
    )

    if token_length(prefix) >= context_length:
        return []

    sentences = split_into_sentences(assistant_text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if current_chunk:
            candidate = current_chunk + " " + sentence
        else:
            candidate = sentence

        if token_length(prefix + candidate) <= context_length:
            current_chunk = candidate

        else:
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = ""

            if token_length(prefix + sentence) <= context_length:
                current_chunk = sentence

            else:
                smaller_chunks = hard_split_text(
                    sentence,
                    prefix,
                    context_length
                )

                chunks.extend(smaller_chunks)

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


def create_training_samples(
    cleaned_data,
    context_length=512
):
    training_samples = []

    stats = {
        "original_conversations": len(cleaned_data),
        "pairs_found": 0,
        "pairs_that_fit": 0,
        "pairs_split": 0,
        "user_prompts_shortened": 0,
        "samples_created": 0,
        "skipped": 0
    }

    for conversation in cleaned_data:
        messages = conversation.get("messages", [])

        i = 0

        while i < len(messages) - 1:
            current = messages[i]
            next_message = messages[i + 1]

            if (
                current.get("role") == "user"
                and next_message.get("role") == "assistant"
            ):
                user_text = current["content"].strip()
                assistant_text = next_message["content"].strip()

                stats["pairs_found"] += 1

                if not user_text or not assistant_text:
                    stats["skipped"] += 1
                    i += 2
                    continue

                user_text, was_shortened = shorten_user_prompt(
                    user_text
                )

                if was_shortened:
                    stats["user_prompts_shortened"] += 1

                full_text = format_pair(
                    user_text,
                    assistant_text
                )

                encoding = fast_tokenizer.encode(full_text)

                if len(encoding.ids) <= context_length:
                    training_samples.append({
                        "text": full_text,
                        "input_ids": encoding.ids,
                        "user": user_text,
                        "assistant": assistant_text,
                        "was_split": False,
                        "user_was_shortened": was_shortened
                    })

                    stats["pairs_that_fit"] += 1

                else:
                    assistant_chunks = split_assistant_response(
                        user_text,
                        assistant_text,
                        context_length
                    )

                    if not assistant_chunks:
                        stats["skipped"] += 1
                        i += 2
                        continue

                    stats["pairs_split"] += 1

                    for chunk in assistant_chunks:
                        text = format_pair(
                            user_text,
                            chunk
                        )

                        encoding = fast_tokenizer.encode(text)

                        if len(encoding.ids) <= context_length:
                            training_samples.append({
                                "text": text,
                                "input_ids": encoding.ids,
                                "user": user_text,
                                "assistant": chunk,
                                "was_split": True,
                                "user_was_shortened": was_shortened
                            })

                i += 2

            else:
                i += 1

    stats["samples_created"] = len(training_samples)

    return training_samples, stats

In [103]:
training_samples, stats = create_training_samples(
    data,
    context_length=512
)

In [104]:
stats

{'original_conversations': 620,
 'pairs_found': 1114,
 'pairs_that_fit': 265,
 'pairs_split': 849,
 'user_prompts_shortened': 5,
 'samples_created': 2745,
 'skipped': 0}

In [105]:
lengths = [
    len(sample["input_ids"])
    for sample in training_samples
]

print("\nTotal training samples:", len(training_samples))
print("Minimum length:", min(lengths))
print("Maximum length:", max(lengths))
print("Average length:", sum(lengths) / len(lengths))

assert max(lengths) <= 512

print("\nAll samples are <= 512 tokens.")


Total training samples: 2745
Minimum length: 52
Maximum length: 512
Average length: 420.28123861566485

All samples are <= 512 tokens.


In [106]:
split_examples = [
    sample
    for sample in training_samples
    if sample["was_split"]
]

print("Split samples:", len(split_examples))

Split samples: 2480


In [107]:
for sample in split_examples[:3]:

    print("=" * 100)

    print("TOKENS:", len(sample["input_ids"]))

    print("\nUSER:")
    print(sample["user"])

    print("\nASSISTANT CHUNK:")
    print(sample["assistant"])

    print()

TOKENS: 507

USER:
The passage highlights that our problems are often not unique and that solutions, much like a 'senpai' or guide, already exist. It prompts us to seek someone who is a few steps ahead. How does one truly recognize such a guide, especially in spiritual matters, and what is the profound significance of submitting to their wisdom in our journey of self-improvement and finding direction, as taught in the Bhagavad Gita?

ASSISTANT CHUNK:
Indeed, the wisdom in seeking guidance is profound and timeless. The Bhagavad Gita emphasizes the crucial role of a spiritual teacher or 'guru' in illuminating our path, much like your 'senpai.' Lord Krishna Himself, in His role as the ultimate guide to Arjuna, lays out the process for acquiring true knowledge. He states in Bhagavad Gita 4.34: (Tad viddhi pranipatena pariprasnena sevaya. Upadeksyanti te jnanam jnaninas tattva-darsinah.)This verse translates to: 'Learn the truth by approaching a spiritual master. Inquire from him submissive

In [108]:
long_users = []

for conversation in data:

    for message in conversation["messages"]:

        if message["role"] == "user":

            text = f"user: {message['content']}\nassistant: "

            length = token_length(text)

            if length > 350:
                long_users.append(length)


print("User prompts > 350 tokens:", len(long_users))

if long_users:
    print("Longest user prefix:", max(long_users))

User prompts > 350 tokens: 7
Longest user prefix: 721


In [109]:
print(training_samples[:2])

[{'text': "user: The passage highlights that our problems are often not unique and that solutions, much like a 'senpai' or guide, already exist. It prompts us to seek someone who is a few steps ahead. How does one truly recognize such a guide, especially in spiritual matters, and what is the profound significance of submitting to their wisdom in our journey of self-improvement and finding direction, as taught in the Bhagavad Gita?\nassistant: Indeed, the wisdom in seeking guidance is profound and timeless. The Bhagavad Gita emphasizes the crucial role of a spiritual teacher or 'guru' in illuminating our path, much like your 'senpai.' Lord Krishna Himself, in His role as the ultimate guide to Arjuna, lays out the process for acquiring true knowledge. He states in Bhagavad Gita 4.34: (Tad viddhi pranipatena pariprasnena sevaya. Upadeksyanti te jnanam jnaninas tattva-darsinah.)This verse translates to: 'Learn the truth by approaching a spiritual master. Inquire from him submissively and r

In [110]:
def add_labels(training_samples):
    processed_samples = []

    for sample in training_samples:
        user_text = sample["user"]
        assistant_text = sample["assistant"]

        prefix = (
            f"user: {user_text}\n"
            f"assistant: "
        )

        full_text = prefix + assistant_text

        full_encoding = fast_tokenizer.encode(full_text)
        prefix_encoding = fast_tokenizer.encode(prefix)

        input_ids = full_encoding.ids.copy()

        labels = input_ids.copy()

        prefix_length = len(prefix_encoding.ids)

        for i in range(prefix_length):
            labels[i] = -100

        processed_samples.append({
            "input_ids": input_ids,
            "labels": labels
        })

    return processed_samples

In [111]:
processed_samples = add_labels(training_samples)

In [112]:
print("Input IDs:")
print(processed_samples[0]["input_ids"])

print("\nLabels:")
print(processed_samples[0]["labels"])

Input IDs:
[451, 264, 58, 32, 375, 568, 115, 294, 902, 104, 552, 320, 380, 673, 366, 273, 589, 112, 405, 822, 109, 115, 470, 335, 116, 355, 426, 340, 105, 603, 423, 366, 772, 108, 912, 279, 839, 989, 512, 324, 39, 115, 276, 568, 105, 39, 32, 284, 32, 103, 117, 297, 101, 44, 769, 495, 809, 455, 105, 304, 416, 637, 280, 112, 673, 117, 262, 275, 336, 101, 107, 859, 514, 642, 445, 289, 382, 119, 775, 501, 115, 259, 959, 100, 305, 488, 422, 432, 514, 542, 117, 525, 298, 515, 103, 110, 105, 122, 256, 115, 117, 330, 289, 103, 117, 297, 388, 367, 112, 827, 582, 269, 924, 307, 293, 117, 97, 327, 348, 986, 115, 496, 594, 301, 268, 637, 335, 435, 115, 344, 110, 634, 440, 287, 362, 335, 283, 117, 98, 109, 379, 316, 275, 483, 321, 115, 100, 479, 560, 273, 589, 106, 816, 364, 263, 335, 283, 480, 102, 45, 296, 112, 405, 749, 109, 476, 274, 102, 764, 316, 878, 298, 99, 974, 279, 44, 259, 262, 410, 635, 261, 389, 66, 281, 478, 118, 97, 257, 71, 293, 97, 63, 10, 97, 684, 105, 304, 287, 116, 58, 32, 73, 

In [113]:
CONTEXT_LENGTH = 512
PAD_TOKEN_ID = 0


def pad_samples(processed_samples, context_length=512):
    padded_samples = []

    for sample in processed_samples:
        input_ids = sample["input_ids"]
        labels = sample["labels"]

        length = len(input_ids)

        padding_length = context_length - length

        padded_input_ids = (
            input_ids
            + [PAD_TOKEN_ID] * padding_length
        )

        attention_mask = (
            [1] * length
            + [0] * padding_length
        )

        padded_labels = (
            labels
            + [-100] * padding_length
        )

        padded_samples.append({
            "input_ids": torch.tensor(
                padded_input_ids,
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention_mask,
                dtype=torch.long
            ),
            "labels": torch.tensor(
                padded_labels,
                dtype=torch.long
            )
        })

    return padded_samples

In [114]:
padded_samples = pad_samples(
    processed_samples,
    context_length=512
)

In [115]:
print(padded_samples[0]["input_ids"].shape)
print(padded_samples[0]["attention_mask"].shape)
print(padded_samples[0]["labels"].shape)

torch.Size([512])
torch.Size([512])
torch.Size([512])


In [116]:
sample = padded_samples[0]

print("Input IDs:")
print(sample["input_ids"])

print("\nAttention mask:")
print(sample["attention_mask"])

print("\nLabels:")
print(sample["labels"])

Input IDs:
tensor([451, 264,  58,  32, 375, 568, 115, 294, 902, 104, 552, 320, 380, 673,
        366, 273, 589, 112, 405, 822, 109, 115, 470, 335, 116, 355, 426, 340,
        105, 603, 423, 366, 772, 108, 912, 279, 839, 989, 512, 324,  39, 115,
        276, 568, 105,  39,  32, 284,  32, 103, 117, 297, 101,  44, 769, 495,
        809, 455, 105, 304, 416, 637, 280, 112, 673, 117, 262, 275, 336, 101,
        107, 859, 514, 642, 445, 289, 382, 119, 775, 501, 115, 259, 959, 100,
        305, 488, 422, 432, 514, 542, 117, 525, 298, 515, 103, 110, 105, 122,
        256, 115, 117, 330, 289, 103, 117, 297, 388, 367, 112, 827, 582, 269,
        924, 307, 293, 117,  97, 327, 348, 986, 115, 496, 594, 301, 268, 637,
        335, 435, 115, 344, 110, 634, 440, 287, 362, 335, 283, 117,  98, 109,
        379, 316, 275, 483, 321, 115, 100, 479, 560, 273, 589, 106, 816, 364,
        263, 335, 283, 480, 102,  45, 296, 112, 405, 749, 109, 476, 274, 102,
        764, 316, 878, 298,  99, 974, 279,  44, 259, 

In [117]:
for sample in padded_samples:
    assert len(sample["input_ids"]) == 512
    assert len(sample["attention_mask"]) == 512
    assert len(sample["labels"]) == 512

print("All samples are padded to 512.")

All samples are padded to 512.


In [118]:
from sklearn.model_selection import train_test_split

train_samples, val_samples = train_test_split(
    padded_samples,
    test_size=0.05,
    random_state=42
)

print("Train samples:", len(train_samples))
print("Validation samples:", len(val_samples))

Train samples: 2607
Validation samples: 138


In [119]:
from torch.utils.data import Dataset


class SFTDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [120]:
train_dataset = SFTDataset(train_samples)
val_dataset = SFTDataset(val_samples)

In [121]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [122]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([32, 512])
torch.Size([32, 512])
torch.Size([32, 512])


### Fine tuning started 

In [123]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

In [124]:
class EmbeddingLayer(nn.Module):

    def __init__(self, vocab_size, context_length, embedding_dim):
        super().__init__()

        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )

        self.position_embedding = nn.Embedding(
            num_embeddings=context_length,
            embedding_dim=embedding_dim
        )

    def forward(self, input_ids):
        batch_size, sequence_length = input_ids.shape

        positions = torch.arange(
            sequence_length,
            device=input_ids.device
        )

        token_embeddings = self.token_embedding(input_ids)
        position_embeddings = self.position_embedding(positions)

        return token_embeddings + position_embeddings

In [125]:
class MultiHeadCausalSelfAttention(nn.Module):

    def __init__(self, d_model, n_heads, context_length):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.query = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.key = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.value = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

        mask = torch.tril(
            torch.ones(
                context_length,
                context_length
            )
        )

        self.register_buffer(
            "mask",
            mask.view(
                1,
                1,
                context_length,
                context_length
            )
        )

    def forward(self, x, attention_mask=None):
        B, T, C = x.shape

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        Q = Q.view(
            B,
            T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            B,
            T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            B,
            T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1)

        scores = scores / (
            self.head_dim ** 0.5
        )

        scores = scores.masked_fill(
            self.mask[:, :, :T, :T] == 0,
            float("-inf")
        )

        if attention_mask is not None:
            padding_mask = attention_mask[:, None, None, :]

            scores = scores.masked_fill(
                padding_mask == 0,
                float("-inf")
            )

        attention_weights = F.softmax(
            scores,
            dim=-1
        )

        out = attention_weights @ V

        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)

        return self.out_proj(out)

In [126]:
class FeedForward(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.fc1 = nn.Linear(
            d_model,
            4 * d_model
        )

        self.fc2 = nn.Linear(
            4 * d_model,
            d_model
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)

        return x


class LayerNorm(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x)


class TransformerBlock(nn.Module):

    def __init__(self, d_model, n_heads, context_length):
        super().__init__()

        self.ln1 = LayerNorm(d_model)

        self.attention = MultiHeadCausalSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            context_length=context_length
        )

        self.ln2 = LayerNorm(d_model)

        self.ffn = FeedForward(d_model)

    def forward(self, x, attention_mask=None):
        x = x + self.attention(
            self.ln1(x),
            attention_mask=attention_mask
        )

        x = x + self.ffn(
            self.ln2(x)
        )

        return x

In [127]:
class SmallLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim,
        num_heads,
        num_layers
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.context_length = context_length

        self.embedding = EmbeddingLayer(
            vocab_size=vocab_size,
            context_length=context_length,
            embedding_dim=embedding_dim
        )

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(
                d_model=embedding_dim,
                n_heads=num_heads,
                context_length=context_length
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(
            embedding_dim
        )

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.embedding.token_embedding.weight
        )

    def forward(self, input_ids, attention_mask=None):
        if input_ids.ndim != 2:
            raise ValueError(
                "input_ids must have shape "
                "[batch_size, sequence_length]"
            )

        if input_ids.size(1) > self.context_length:
            raise ValueError(
                f"Sequence length {input_ids.size(1)} exceeds "
                f"context length {self.context_length}"
            )

        x = self.embedding(input_ids)

        for block in self.transformer_blocks:
            x = block(
                x,
                attention_mask=attention_mask
            )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits

In [128]:
vocab_size = 1000
context_length = 512
embedding_dim = 96
num_heads = 6
num_layers = 4

In [129]:
vocab_size = 1000
context_length = 512
embedding_dim = 96
num_heads = 6
num_layers = 4

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = SmallLanguageModel(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers
).to(device)

print("Device:", device)

Device: cuda


In [130]:
model_path = Path("../model/saved_slm/model_weights.pt")

if not model_path.exists():
    model_path = Path("saved_slm/model_weights.pt")

if not model_path.exists():
    raise FileNotFoundError(
        f"Model weights not found: {model_path.resolve()}"
    )

state_dict = torch.load(
    model_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(state_dict)

print("Pretrained model loaded")

Pretrained model loaded


In [131]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

with torch.no_grad():
    logits = model(
        input_ids,
        attention_mask=attention_mask
    )

print("Input shape:", input_ids.shape)
print("Logits shape:", logits.shape)

Input shape: torch.Size([32, 512])
Logits shape: torch.Size([32, 512, 1000])


In [132]:
learning_rate = 5e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.01
)

loss_function = nn.CrossEntropyLoss(
    ignore_index=-100
)

In [133]:
def train_epoch(
    model,
    train_loader,
    optimizer,
    loss_function,
    device
):
    model.train()

    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            input_ids,
            attention_mask=attention_mask
        )

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        loss = loss_function(
            shift_logits.view(
                -1,
                model.vocab_size
            ),
            shift_labels.view(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [134]:
@torch.no_grad()
def evaluate(
    model,
    val_loader,
    loss_function,
    device
):
    model.eval()

    total_loss = 0.0

    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            input_ids,
            attention_mask=attention_mask
        )

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        loss = loss_function(
            shift_logits.view(
                -1,
                model.vocab_size
            ),
            shift_labels.view(-1)
        )

        total_loss += loss.item()

    return total_loss / len(val_loader)

In [135]:
num_epochs = 100000

best_val_loss = float("inf")

save_directory = Path(
    "fine_tuned_slm"
)

save_directory.mkdir(
    parents=True,
    exist_ok=True
)

best_model_path = (
    save_directory
    / "best_model_weights.pt"
)

checkpoint_path = save_directory / "checkpoint.pt"

start_epoch = 0
resume_batch = 0
resume_loss = 0.0

temporary_checkpoint_path = checkpoint_path.with_suffix(".pt.tmp")
checkpoint = None
checkpoint_source = None
checkpoint_candidates = [checkpoint_path, temporary_checkpoint_path]
checkpoint_candidates = sorted(
    (path for path in checkpoint_candidates if path.exists()),
    key=lambda path: path.stat().st_mtime,
    reverse=True
)

for candidate_path in checkpoint_candidates:
    if candidate_path.stat().st_size == 0:
        print(f"Skipping empty checkpoint: {candidate_path}")
        continue
    try:
        checkpoint = torch.load(
            candidate_path,
            map_location=device,
            weights_only=False
        )
        checkpoint_source = candidate_path
        break
    except (EOFError, RuntimeError, OSError) as error:
        print(f"Skipping invalid checkpoint {candidate_path}: {error}")

if checkpoint is not None:
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    # Older checkpoints were saved only after a complete epoch.
    if "batch" in checkpoint:
        start_epoch = checkpoint["epoch"]
        resume_batch = checkpoint["batch"]
        resume_loss = checkpoint.get("running_loss", 0.0)
    else:
        start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]

    print("Checkpoint loaded from:", checkpoint_source)
    print("Resuming from epoch:", start_epoch + 1)
    print("Resuming after batch:", resume_batch)

else:
    print("No checkpoint found")
    print("Starting fine-tuning from beginning")

def save_checkpoint_safely(checkpoint):
    torch.save(checkpoint, temporary_checkpoint_path)
    try:
        temporary_checkpoint_path.replace(checkpoint_path)
    except PermissionError:
        # Windows may temporarily lock checkpoint.pt. The complete .tmp
        # file is retained and will be preferred when training resumes.
        print("checkpoint.pt is locked; progress saved to:",
              temporary_checkpoint_path)

for epoch in range(start_epoch, num_epochs):

    # Recreate the same shuffled order for this epoch after a restart.
    epoch_generator = torch.Generator()
    epoch_generator.manual_seed(42 + epoch)
    epoch_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        generator=epoch_generator
    )

    model.train()
    total_loss = resume_loss if epoch == start_epoch else 0.0
    first_batch = resume_batch if epoch == start_epoch else 0

    for batch_index, batch in enumerate(epoch_loader):
        if batch_index < first_batch:
            continue

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids, attention_mask=attention_mask)
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss = loss_function(
            shift_logits.view(-1, model.vocab_size),
            shift_labels.view(-1)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

        # Save after every completed batch. batch_index + 1 is the
        # number of batches to skip when this epoch is resumed.
        save_checkpoint_safely({
            "epoch": epoch,
            "batch": batch_index + 1,
            "running_loss": total_loss,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss
        })

    train_loss = total_loss / len(epoch_loader)

    val_loss = evaluate(
        model,
        val_loader,
        loss_function,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Validation Loss: {val_loss:.4f}"
    )

    print("-" * 50)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("Best model saved")

    # Mark the epoch complete; the next run starts at the next epoch.
    save_checkpoint_safely({
        "epoch": epoch + 1,
        "batch": 0,
        "running_loss": 0.0,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss
    })

    print("Epoch checkpoint saved")

    resume_batch = 0
    resume_loss = 0.0

Checkpoint loaded from: fine_tuned_slm\checkpoint.pt.tmp
Resuming from epoch: 3
Resuming after batch: 76
Epoch 3/100000
Train Loss: 2.9307
Validation Loss: 2.9542
--------------------------------------------------
Best model saved
Epoch checkpoint saved
Epoch 4/100000
Train Loss: 2.7946
Validation Loss: 2.8512
--------------------------------------------------
Best model saved
Epoch checkpoint saved
Epoch 5/100000
Train Loss: 2.7010
Validation Loss: 2.7741
--------------------------------------------------
Best model saved
Epoch checkpoint saved
checkpoint.pt is locked; progress saved to: fine_tuned_slm\checkpoint.pt.tmp
Epoch 6/100000
Train Loss: 2.6272
Validation Loss: 2.7147
--------------------------------------------------
Best model saved
Epoch checkpoint saved
Epoch 7/100000
Train Loss: 2.5650
Validation Loss: 2.6662
--------------------------------------------------
Best model saved
Epoch checkpoint saved
Epoch 8/100000
Train Loss: 2.5183
Validation Loss: 2.6249
---------------

KeyboardInterrupt: 

In [151]:
# Load the weights from the epoch with the lowest validation loss.
best_model_path = Path("fine_tuned_slm/best_model_weights.pt")

if not best_model_path.exists() or best_model_path.stat().st_size == 0:
    raise FileNotFoundError(
        f"No valid best-model weights found at {best_model_path.resolve()}"
    )

best_model = SmallLanguageModel(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers
).to(device)

best_model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device,
        weights_only=True
    )
)
best_model.eval()
print("Best validation-loss model loaded from:", best_model_path.resolve())


@torch.inference_mode()
def predict(
    question,
    max_new_tokens=150,
    temperature=0.7,
    top_k=40
):
    prompt = f"user: {question.strip()}\nassistant: "
    prompt_ids = fast_tokenizer.encode(prompt).ids

    if len(prompt_ids) >= context_length:
        prompt_ids = prompt_ids[-(context_length - 1):]

    generated = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device
    )

    generated_answer_ids = []

    for _ in range(max_new_tokens):
        model_input = generated[:, -context_length:]
        logits = best_model(model_input)[:, -1, :]

        if temperature <= 0:
            next_token = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            k = min(top_k, logits.size(-1))
            top_values, top_indices = torch.topk(logits, k=k, dim=-1)
            probabilities = torch.softmax(top_values, dim=-1)
            sampled_position = torch.multinomial(probabilities, 1)
            next_token = top_indices.gather(-1, sampled_position)

        token_id = next_token.item()
        generated_answer_ids.append(token_id)
        generated = torch.cat((generated, next_token), dim=1)

        answer_so_far = fast_tokenizer.decode(generated_answer_ids)
        if "\nuser:" in answer_so_far:
            break

    answer = fast_tokenizer.decode(generated_answer_ids)
    return answer.split("\nuser:", 1)[0].strip()

Best validation-loss model loaded from: C:\Users\Tvari\Desktop\TvaritRepo\SLM\slm-from-scratch\model\fine_tuned_slm\best_model_weights.pt


In [143]:
question = "The passage highlights that our problems are often not unique and that solutions, much like a 'senpai' or guide, already exist. It prompts us to seek someone who is a few steps ahead. How does one truly recognize such a guide, especially in spiritual matters, and what is the profound significance of submitting to their wisdom in our journey of self-improvement and finding direction, as taught in the Bhagavad Gita?"


In [150]:
print(predict(question, temperature=0.01, 
max_new_tokens=200, top_k=100))

Indeed, your question delves into the very essence of Karma Yoga, which is not merely external circumstances. Lord Krishna Himself declares in the Bhagavad Gita (2.14): " " This means, "Work done as a sacrifice for Vishnu has to be performed; otherwise, work binds us to perform your actions become sincerity and death.' The passage highlights that the senance of happiness to subtle the senses of material des
